# Day 11 — Unified Recommendation Engine & Decision Intelligence (Layer 11)

The **orchestration layer** — where nine independent intelligence modules become one
decision-making system. It aggregates every prior layer (Days 4–10) into a single
**VentureDecision**: a readiness verdict, prioritized evidence-backed actions, a transparent
confidence score, and separate founder & investor strategies — all on **one warm faiss-free stack**.

**Honesty.** Deterministic aggregation, **no LLM, no new model**. Recommendations come only from
real controllable levers (goal / duration / launch timing / title) and category-cohort evidence;
retention / revenue / moat / team / pricing / growth-trajectory are **deferred**, not fabricated.
"Similar startups" = the same-category cohort in the shared item frame (no semantic retrieval).
Model-simulated, **not causal**. Logic in `src/recommendation/`.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# ONE warm faiss-free stack orchestrating Days 4-10 (copilot + evolution + scenario + counterfactual).
from src.recommendation.pipeline import RecommendationPipeline
pipe = RecommendationPipeline()
assert pipe.agg.cf is pipe.confidence.agg.cf is pipe.investor.agg.cf     # one model, every sub-engine

df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][df.iloc[ti]["usd_goal_real"] > 12000].sample(1, random_state=4).iloc[0].to_dict()
vi = pipe.agg.aggregate(row)               # Phase B: the aggregated intelligence profile
D  = pipe.run(row)                          # the full VentureDecision

ir = D["investment_readiness"]
print("CAMPAIGN: %s | %s | goal $%.0f" % (row["name"], row["main_category"], row["usd_goal_real"]))
print("DECISION: %s | readiness index %d | confidence %.0f%%" %
      (D["investor_strategy"]["recommendation"], ir["readiness_index"], D["confidence"]["score"]*100))

CAMPAIGN: Artificer - Board Game | Games | goal $30516
DECISION: Proceed with Caution | readiness index 45 | confidence 83%


## Phase B — Intelligence Aggregation

Every prior layer's output, collected into one profile on a single loaded model. "Similar startups"
is the same-category cohort in the pandas item frame — no faiss, no torch.

In [2]:
print("Aggregated from every prior layer (one warm stack, no faiss):")
print("  prediction  (D4) :", "%.0f%%" % (vi["prediction"]["funding_probability"]*100), vi["prediction"]["predicted_label"])
print("  drivers     (D5) :", [d["feature"] for d in vi["drivers"]["top_positive"][:3]], "(+)")
print("  counterfactual(D8):", len(vi["counterfactual"]["ranked_recommendations"]), "ranked recommendations")
print("  copilot     (D9) :", list(vi["copilot"].keys()))
print("  evolution   (D10):", "stage", vi["evolution"]["stage"]["stage"])
print("  scenario    (D10):", "best %.0f%%" % (vi["scenario"]["best_scenario"]["probability"]*100))
print("  cohort           :", vi["cohort"]["main_category"],
      "n=%d, funded %.0f%% vs %.0f%% baseline" %
      (vi["cohort"]["n_similar"], vi["cohort"]["cohort_funding_rate"]*100, vi["cohort"]["platform_base_rate"]*100))

Aggregated from every prior layer (one warm stack, no faiss):
  prediction  (D4) : 24% failed
  drivers     (D5) : ['category_freq', 'main_category_Games', 'uppercase_ratio'] (+)
  counterfactual(D8): 5 ranked recommendations
  copilot     (D9) : ['investor_objections', 'risk_analysis', 'swot', 'category_analysis', 'competition_analysis', 'recommendations']
  evolution   (D10): stage Fixable
  scenario    (D10): best 69%
  cohort           : Games n=22878, funded 44% vs 40% baseline


## Phase C — Strategic Recommendations

Categorized, evidence-backed recommendations from the real levers. Product / team / financial / tech
/ GTM strategies are deferred (no data). Category positioning is observational context, not an action.

In [3]:
for r in D["recommendations"]:
    print("[%-20s] %-9s %+dpts  %s" % (r["category"], r["priority_label"], r["expected_impact_points"], r["action"]))
    print("     evidence: %s" % (r["evidence"] or "")[:82])
print("\npositioning:", D["category_positioning"]["insight"])

[Campaign Strategy   ] Critical  +30pts  Shorten your campaign from 29 to 15 days
     evidence: Games campaigns with a <=15d campaign funded at 66% vs a 40% baseline (lift 1.62, 
[Fundraising Strategy] High      +12pts  Lower your funding goal from $30,516 to $5,000
     evidence: Games campaigns asking goal <= $5,000 funded at 55% vs a 40% baseline (lift 1.36, 
[Campaign Strategy   ] Medium    +4pts  Add a number to your title (e.g. an edition or volume number)
     evidence: Games campaigns with a number in the title funded at 59% vs a 40% baseline (lift 1
[Campaign Strategy   ] Medium    +3pts  Launch on Tuesday instead of Thursday
     evidence: Games campaigns launched on Tuesday funded at 50% vs a 40% baseline (lift 1.24, n=

positioning: Games campaigns fund at 44% vs the 40% platform baseline — a favorable category to launch in (n=22,878).


## Phase D — Prioritization (transparent cross-layer score)

Ranked by a weighted mean of six factors from four layers — Day-8 impact, Day-7 evidence, Day-9
investor overlap, Day-10 scenario gain, feasibility, confidence. Every factor is shown, so the
ranking is auditable.

In [4]:
print("rk feature            score | impact evid invOvl scenGain feas conf")
for a in D["priority_improvements"]:
    f = a["factors"]
    print("#%d %-18s %.3f |  %.2f  %.2f  %.2f    %.2f    %.2f %.2f" %
          (a["rank"], a["feature"], a["priority_score"], f["impact"], f["evidence"],
           f["investor_overlap"], f["scenario_gain"], f["feasibility"], f["confidence"]))
print("\nweights:", D["priority_improvements"] and pipe.prioritizer.weights)

rk feature            score | impact evid invOvl scenGain feas conf
#1 campaign_duration  0.850 |  1.00  1.00  0.00    1.00    1.00 1.00
#2 goal               0.658 |  0.48  0.60  1.00    0.47    1.00 1.00
#3 title              0.334 |  0.16  0.76  0.00    0.17    0.80 0.60
#4 launch_weekday     0.244 |  0.12  0.40  0.00    0.11    0.90 0.60

weights: {'impact': 0.3, 'evidence': 0.2, 'investor_overlap': 0.15, 'scenario_gain': 0.2, 'feasibility': 0.05, 'confidence': 0.1}


## Phase G — Confidence (the new core)

How much to trust the recommendation — a transparent blend of **real agreement signals**, not the
prediction itself. Note the cross-method signal (`cf_stability`: Day-8 greedy optimum vs Day-10 joint
best) and `model_agreement` (the three Day-4 models on one shared feature transform).

In [5]:
c = D["confidence"]
print("CONFIDENCE  %.2f\n" % c["score"])
for k, v in c["factors"].items():
    print("  %-20s %.2f" % (k, v))
print("\nreason:", c["reason"])

CONFIDENCE  0.83

  prediction_margin    0.29
  cohort_agreement     0.56
  pattern_support      0.67
  scenario_stability   1.00
  cf_stability         0.89
  model_agreement      1.00

reason: Supported by strong historical pattern support, stable scenario behavior, consistent reachability across methods and agreement across models. Main uncertainty: a clear prediction margin.


## Phase E — Founder Strategy (view 1)

The founder-facing projection: a pre-launch action plan anchored to the readiness stage.

In [6]:
fs = D["founder_strategy"]
print("HEADLINE:", fs["headline"])
print("FOCUS FIRST:", fs["focus_first"], "\n")
for a in fs["action_plan"]:
    flag = "  [investor concern]" if a["addresses_investor_concern"] else ""
    print("  %d. [%-9s] %s%s" % (a["step"], a["priority_label"], a["action"], flag))

HEADLINE: Fixable before launch. As-is the model estimates 24%, but the actions below can reach ~66% — above the 33% bar. Do them before you go live.
FOCUS FIRST: ['Shorten your campaign from 29 to 15 days', 'Lower your funding goal from $30,516 to $5,000'] 

  1. [Critical ] Shorten your campaign from 29 to 15 days
  2. [High     ] Lower your funding goal from $30,516 to $5,000  [investor concern]
  3. [Medium   ] Add a number to your title (e.g. an edition or volume number)
  4. [Medium   ] Launch on Tuesday instead of Thursday


## Phase F — Investor Strategy (view 2)

The investor-facing projection — one of four bands from the 2-D readiness stage + reachable headroom,
optionally downgraded by low confidence. Same intelligence as the founder view, so they can't
contradict each other.

In [7]:
iv = D["investor_strategy"]
print("RECOMMENDATION:", iv["recommendation"])
print("reason        :", iv["reason"])
print("key risk      :", iv["key_risk"]["reason"])
print("reconsider if :", iv["conditions_to_reconsider"])

RECOMMENDATION: Proceed with Caution
reason        : Below the bar as-is (24%) but with strong reachable upside to ~66% if the team executes the priority changes. Main risk: Goal $30,516 is 6.1x the Games funded-median ($5,000); large goals are harder to reach.
key risk      : Goal $30,516 is 6.1x the Games funded-median ($5,000); large goals are harder to reach.
reconsider if : ['Team addresses the top concern (goal_size).', 'Campaign reaches its reachable readiness (~66%) before launch.']


## The unified decision — readiness, risks, opportunities, final strategy

In [8]:
print("FINAL STRATEGY:", D["final_strategy"], "\n")
print("readiness : %s | index %d | %.0f%% -> %.0f%% (bar %.0f%%) | headroom +%.0f pts" %
      (ir["stage"], ir["readiness_index"], ir["funding_probability"]*100,
       ir["optimized_probability"]*100, ir["threshold"]*100, D["improvement_headroom"]*100))
print("risks     :", D["risks"]["overall_risk_level"], "-", ", ".join(r["category"] for r in D["risks"]["top_risks"]))
print("opportunities:")
for o in D["opportunities"]:
    print("   -", o["detail"])

FINAL STRATEGY: Proceed with Caution: Fixable at 24% (reachable ~66%), confidence 83%. Founder priority: shorten your campaign from 29 to 15 days; lower your funding goal from $30,516 to $5,000. 

readiness : Fixable | index 45 | 24% -> 66% (bar 33%) | headroom +42 pts
risks     : High - financial_goal_risk, funding_risk, cohort_risk
opportunities:
   - Priority changes can lift funding odds from 24% to ~66%.
   - Games funds at 44% vs the 40% baseline — a favorable category.
   - Working in your favor: category_freq.
   - Working in your favor: main_category_Games.


## The honesty boundary

The orchestrator inherits and preserves every layer's deferrals — what it refuses to fabricate.

In [9]:
print("DEFERRED — refused, no data to support (%d items):" % len(D["deferred"]))
items = list(D["deferred"])
for k in items[:12]:
    print("   -", k)
print("   ... and %d more" % (len(items) - 12))

DEFERRED — refused, no data to support (37 items):
   - customer_retention
   - monthly_revenue
   - revenue_growth
   - unit_economics
   - competitive_moat
   - named_competitors
   - founder_experience
   - team_size
   - pricing_strategy
   - market_size_tam
   - product_traction
   - funding_rounds
   ... and 25 more


## Phase I — Serialization (content-hash identity + verify)

The engine is a decision **policy** over the Day-4 model + Day-8/9/10 engines — persisted as JSON,
not a pickle. Its content hash pins the policy; the dependency hashes form a Merkle-style chain over
the whole Day-4→10 stack.

In [10]:
from src.recommendation.serialization import RecommendationSerializer
ser = RecommendationSerializer().build(); ser.save()
v = ser.verify()
print(ser.describe())
print("verify all_ok:", v["all_ok"], "(%d/%d checks pass)" % (sum(v["checks"].values()), len(v["checks"])))

Recommendation engine v1.0.0 (content b3a5439f5991…): layer 11; 4 deps hashed, 7 artifacts, 18 items deferred.
verify all_ok: True (12/12 checks pass)


## Day 11 complete — Layer 11 ✅

The platform is now a **decision-intelligence system**, not a collection of models. Every capability
from Days 4–10 feeds one evidence-backed `VentureDecision` with a readiness verdict, prioritized
actions, transparent confidence, and consistent founder & investor strategies — on one warm
faiss-free stack, content-hash verified. Retention, revenue, moat, team, and growth remain
**deferred, not invented**. Model-simulated, not causal.

Foundation for **Layer 12 — Multi-Agent Intelligence** and **Layer 13 — Professional Reports**.